# Multi-Agent Pricer

FrontierAgent (OpenAI RAG + ChromaDB), Gemini, and Claude (via OpenRouter) ensemble.
Includes a scripted planner and an autonomous LLM planner with tool calling.
Gradio UI.

In [ ]:
import os
import sys
import json
import re
import logging
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
import gradio as gr

os.chdir(os.path.join(os.path.abspath(""), "..", ".."))

from agents.agent import Agent
from agents.scanner_agent import ScannerAgent
from agents.frontier_agent import FrontierAgent
from agents.messaging_agent import MessagingAgent
from agents.deals import Deal, Opportunity
from deal_agent_framework import DealAgentFramework

load_dotenv(override=True)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

for name, key in [("OpenAI", openai_api_key), ("Google", google_api_key), ("OpenRouter", openrouter_api_key)]:
    print(f"{name}: {'set' if key else 'NOT set'}")

In [ ]:
# Separate clients for each provider (missing keys are warned, not fatal)
openai_client = OpenAI() if openai_api_key else None
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") if google_api_key else None
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1") if openrouter_api_key else None

if not openai_client:
    print("WARNING: OPENAI_API_KEY not set — FrontierAgent will not work")
if not gemini_client:
    print("WARNING: GOOGLE_API_KEY not set — GeminiAgent will be skipped")
if not openrouter_client:
    print("WARNING: OPENROUTER_API_KEY not set — ClaudeAgent will be skipped")

In [ ]:
def get_price(s: str) -> float:
    s = (s or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(m.group()) if m else 0.0


def _chat_price(client: OpenAI, model: str, description: str) -> float:
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Estimate the price in USD. Reply with only the number.\n\n{description}"}],
        max_tokens=32,
    )
    return get_price(r.choices[0].message.content or "")


class GeminiAgent(Agent):
    name, color = "Gemini", Agent.MAGENTA
    def __init__(self, model: str = "gemini-2.0-flash"):
        self.client = gemini_client
        self.model = model
    def price(self, description: str) -> float:
        return _chat_price(self.client, self.model, description)


class ClaudeAgent(Agent):
    name, color = "Claude", Agent.CYAN
    def __init__(self, model: str = "anthropic/claude-3.5-haiku"):
        self.client = openrouter_client
        self.model = model
    def price(self, description: str) -> float:
        return _chat_price(self.client, self.model, description)

In [ ]:
# ChromaDB + agents (only create agents whose clients are available)
DB = "products_vectorstore"
collection = chromadb.PersistentClient(path=DB).get_or_create_collection("products")

frontier = FrontierAgent(collection) if openai_client else None
gemini_agent = GeminiAgent() if gemini_client else None
claude_agent = ClaudeAgent() if openrouter_client else None

agents_and_weights = []
if frontier:
    agents_and_weights.append((frontier, 0.6))
if gemini_agent:
    agents_and_weights.append((gemini_agent, 0.2))
if claude_agent:
    agents_and_weights.append((claude_agent, 0.2))

_total = sum(w for _, w in agents_and_weights)
agents_and_weights = [(a, w / _total) for a, w in agents_and_weights]

print(f"Ensemble: {[(a.name, f'{w:.0%}') for a, w in agents_and_weights]}")


def ensemble_price(description: str) -> float:
    if not agents_and_weights:
        return 0.0
    return sum(a.price(description) * w for a, w in agents_and_weights)

In [ ]:
class PlanningAgentLLM(Agent):
    """Scripted planner: scan -> ensemble price -> sort -> alert."""
    name, color, DEAL_THRESHOLD = "Planning (LLM)", Agent.GREEN, 50

    def __init__(self, collection):
        self.scanner = ScannerAgent()
        self.messenger = MessagingAgent()
        self._collection = collection

    def run(self, deal: Deal) -> Opportunity:
        est = ensemble_price(deal.product_description)
        return Opportunity(deal=deal, estimate=est, discount=est - deal.price)

    def plan(self, memory=None):
        memory = memory or []
        sel = self.scanner.scan(memory=memory)
        if not sel:
            return None
        opps = [self.run(d) for d in sel.deals[:5]]
        opps.sort(key=lambda o: o.discount, reverse=True)
        best = opps[0]
        if best.discount > self.DEAL_THRESHOLD:
            self.messenger.alert(best)
        return best if best.discount > self.DEAL_THRESHOLD else None

In [ ]:
# Test the scripted planner
framework = DealAgentFramework()
framework.planner = PlanningAgentLLM(framework.collection)
opportunities = framework.run()
print(f"{len(opportunities)} deals in memory. Latest: {opportunities[-1] if opportunities else None}")

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "scan",
        "description": "Scan the internet for current deals and bargains.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "estimate",
        "description": "Estimate the true market value of a product given its description.",
        "parameters": {"type": "object", "properties": {
            "description": {"type": "string", "description": "Product description"}
        }, "required": ["description"]},
    }},
    {"type": "function", "function": {
        "name": "alert",
        "description": "Send a push notification about a great deal to the user.",
        "parameters": {"type": "object", "properties": {
            "description": {"type": "string"},
            "deal_price": {"type": "number"},
            "estimated_value": {"type": "number"},
            "url": {"type": "string"},
        }, "required": ["description", "deal_price", "estimated_value", "url"]},
    }},
]


class AutonomousPlannerLLM(Agent):
    """Autonomous planner that lets the LLM decide which tools to call and in what order."""
    name, color = "Autonomous Planner", Agent.YELLOW
    MODEL = "gpt-4o-mini"
    SYSTEM = (
        "You are a deal-hunting assistant. You have three tools: scan, estimate, and alert. "
        "First scan for deals. Then estimate each deal's true value. "
        "If any deal has a discount (estimated value minus deal price) greater than $50, "
        "alert the user about the single best deal. Then stop."
    )

    def __init__(self):
        self.scanner = ScannerAgent()
        self.messenger = MessagingAgent()
        self.llm = openai_client
        self.memory = []
        self.opportunity = None

    def _handle_call(self, name: str, args: dict) -> str:
        self.log(f"Tool call: {name}")
        if name == "scan":
            sel = self.scanner.scan(memory=self.memory)
            return sel.model_dump_json() if sel else "No deals found."
        if name == "estimate":
            description = (args or {}).get("description", "")
            if not description:
                return "Missing required field: description"
            val = ensemble_price(description)
            return f"${val:.2f}"
        if name == "alert":
            try:
                desc = (args or {}).get("description", "")
                price = float((args or {}).get("deal_price", 0))
                est = float((args or {}).get("estimated_value", 0))
                url = (args or {}).get("url", "")
                if not desc or not url:
                    return "Missing required fields for alert"
                self.messenger.notify(desc, price, est, url)
                self.opportunity = Opportunity(
                    deal=Deal(product_description=desc, price=price, url=url),
                    estimate=est,
                    discount=est - price,
                )
                return "Notification sent."
            except Exception as e:
                return f"alert failed: {e}"
        return "Unknown tool."

    def plan(self, memory=None):
        self.memory = memory or []
        self.opportunity = None
        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user", "content": "Go ahead — scan, estimate, and alert if there's a good deal."},
        ]
        turns = 0
        max_turns = 30
        while True:
            turns += 1
            if turns > max_turns:
                self.log(f"Stopping autonomous loop after {max_turns} turns")
                break
            resp = self.llm.chat.completions.create(
                model=self.MODEL, messages=messages, tools=TOOLS,
            )
            choice = resp.choices[0]
            if choice.finish_reason != "tool_calls":
                self.log(f"LLM finished: {choice.message.content}")
                break
            messages.append(choice.message)
            for tc in choice.message.tool_calls:
                args = json.loads(tc.function.arguments or "{}")
                result = self._handle_call(tc.function.name, args)
                messages.append({"role": "tool", "content": result, "tool_call_id": tc.id})
        return self.opportunity

In [ ]:
# Test the autonomous planner
auto = AutonomousPlannerLLM()
result = auto.plan([])
print(result)

In [ ]:
import logging
import queue
import threading
import time
from log_utils import reformat


class QueueHandler(logging.Handler):
    def __init__(self, log_queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))


def html_for(log_data):
    output = "<br>".join(log_data[-18:])
    return f'<div style="min-height:400px;height:100%;overflow-y:auto;border:1px solid #ccc;background:#222229;padding:10px;">{output}</div>'


def setup_logging(log_queue):
    handler = QueueHandler(log_queue)
    handler.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S %z"))
    logger = logging.getLogger()
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    return handler


try:
    app_fw = framework
except NameError:
    app_fw = DealAgentFramework()

app_fw.planner = PlanningAgentLLM(app_fw.collection)


def table_for(opps):
    return [
        [o.deal.product_description, f"${o.deal.price:.2f}", f"${o.estimate:.2f}", f"${o.discount:.2f}", o.deal.url]
        for o in (opps or [])
    ]


def update_output(log_data, log_queue, result_queue):
    initial_result = table_for(app_fw.memory)
    final_result = None
    while True:
        try:
            message = log_queue.get_nowait()
            log_data.append(reformat(message))
            yield log_data, html_for(log_data), final_result or initial_result
        except queue.Empty:
            try:
                final_result = result_queue.get_nowait()
                yield log_data, html_for(log_data), final_result or initial_result
            except queue.Empty:
                if final_result is not None:
                    break
                time.sleep(0.1)


def run_with_logging(initial_log_data):
    log_queue = queue.Queue()
    result_queue = queue.Queue()
    handler = setup_logging(log_queue)

    def worker():
        result_queue.put(table_for(app_fw.run()))

    thread = threading.Thread(target=worker)
    thread.start()

    for ld, html, tbl in update_output(initial_log_data, log_queue, result_queue):
        yield ld, html, tbl

    logging.getLogger().removeHandler(handler)


with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    log_data = gr.State([])

    gr.Markdown("**##The Price is Right** – FrontierAgent RAG + Gemini + Claude")
    df = gr.Dataframe(headers=["Description", "Price", "Estimate", "Discount", "URL"], wrap=True, max_height=400)
    run_btn = gr.Button("Run cycle")
    logs = gr.HTML()
    run_btn.click(run_with_logging, inputs=[log_data], outputs=[log_data, logs, df])
    ui.load(run_with_logging, inputs=[log_data], outputs=[log_data, logs, df])

ui.launch(inbrowser=True)